# PROC: feedback V4 check (business comparison with Every Angle, 16 Nov 2026)

**17 Nov 2026.** The business compared `Fact_PO_LineItem_Schedule` (294,045 rows, POs created since 1 Sep 2026) with Every Angle and
listed duplicates, NULL schedule lines, different execution statuses, missing POs and empty GR quantities
(`Procurment/Fabric_output_V4_20261116.xlsx`; our reading of every difference, groups D1 to D12, is in
`Procurment/Fabric_output_V4_findings.xlsx`). This notebook holds the queries that settle the open groups. It only reads data and
changes nothing; the results stay with the notebook. Run it in the Procurement workspace after Bronze, Silver and `NB_Gold_PROC`.

| Query | Settles | Reads |
|---|---|---|
| Q1 | D1: how many lines have no goods receipt in Fabric although Every Angle shows one | Silver EKET, EKPO, EKKO, EKBE |
| Q2 | D1 / D10: do Bronze EKET and EKBE still receive changes | Bronze, OPS `table_log` |
| Q3 | D1: the example POs through every layer; at which layer the receipt disappears | Bronze → Silver → Gold, DAB source if `SOURCE_ABFSS` is set |
| Q4 | D1: a key problem between EKBE and EKPO | Silver |
| Q5 | D3: zero-quantity schedule lines, and the ones that show a receipt | Silver EKET, EKBE |
| Q6 | D5: Closed by GR where Every Angle says Outbound delivery created (stock transfers) | Gold, Silver, SD Silver LIPS |
| Q7 | D6: the four open-line status pairs | Gold, Silver EKES / EKET / EKPO |
| Q8 | D10: the PO Every Angle has and Fabric lacks | Bronze EKKO, EKPO, EKET |
| Q9 | D7 / D8: duplicated keys and over-confirmed pieces (0 in the fact after 17 Nov) | Gold |
| Q10 | D2: how many lines would change with a line-level execution status | Gold |
| Q12 | Line-level comparison with the 16 Nov extract (19 Nov 2026): the extract's Every Angle values (uploaded file) against today's order and execution status and the execution status with Option A; prints totals, per-cause tables and example lines; a table only when `SAVE_COMPARE_TABLE` is True | Uploaded `EA_FILE`, Gold, Silver EKET / EKPO / EKBE |

Decisions D7 and D8 were taken on 17 Nov 2026 (one row per schedule line; over-confirmed pieces only in the evidence table).
D2 to D6 wait for the business; D1 and D10 are data questions for Amarildo and team.


In [1]:
# ── Parameters ─────────────────────────────────────────────────────────────────────────────────────────────
PIPELINE_RUN_ID = ''
EXAMPLE_POS     = ['13219558', '13219560', '13219566', '13219607', '13219824', '13219834', '13148876', '4508252311']
CREATED_FROM    = '2026-09-01'      # the business extract: POs created on or after this date (EKKO AEDAT)
EXTRACT_TO      = '2026-11-14'      # ▶ 19 Nov 2026: last creation date in the 16 Nov extract (Q12 exports the same POs)
EA_FILE         = 'Files/PROC_EA_16Nov_upload.csv'   # ▶ 19 Nov 2026: Q12 input, uploaded to the PROC Gold lakehouse Files
N_LINE_EXAMPLES = 10                # ▶ 19 Nov 2026: Q12 example lines printed per result and cause
SAVE_COMPARE_TABLE = False          # ▶ 19 Nov 2026: True writes Q12's line list to Gold table Chk_PROC_Order_Status_Compare (for SQL)
SOURCE_ABFSS    = ''                # optional: abfss path of the DAB lakehouse that feeds PROC Bronze, to compare EKET / EKBE at the source
SOURCE_SCHEMA   = 'dbo'
N_EXAMPLES      = 20

StatementMeta(, d128d76f-41af-48d1-bc01-8de74e6840cc, 3, Finished, Available, Finished, False)

In [2]:
%run ./NB_Utils_Common

StatementMeta(, d128d76f-41af-48d1-bc01-8de74e6840cc, 9, Finished, Available, Finished, True)

NB_Utils_Common: settings ready
NB_Utils_Common: locations ready
NB_Utils_Common: start_run() ready
NB_Utils_Common: OPS logging ready
NB_Utils_Common: dq_record(), dq_flush(), check_volume_change() ready
NB_Utils_Common loaded: start_run(), tpath(), record_table(), dq_record(), end_of_run_report(), end_session()


In [3]:
%run ./NB_Config_PROC

StatementMeta(, d128d76f-41af-48d1-bc01-8de74e6840cc, 16, Finished, Available, Finished, True)

Config loaded: ['EKKO', 'EKPO', 'EKET', 'EKBE']
EA audit report tables added: EKES, EKKN, EKPA, EKAB
New tables added:EKPV, EBKN, EBAN


In [4]:
%run ./NB_Utils_Gold

StatementMeta(, d128d76f-41af-48d1-bc01-8de74e6840cc, 17, Finished, Available, Finished, True)

NB_Utils_Gold loaded: silver(), gold(), drop_redundant(), sk_lookup(), write_gold(), stable_sk(), promote_to_gold(), profile_strings()


In [5]:
# ── Setup ──────────────────────────────────────────────────────────────────────────────────────────────────
start_run('PROC_Check_Feedback_V4', 'CHECK')
fact  = gold('Fact_PO_LineItem_Schedule')
ev    = gold('Fact_PO_Order_Status_Evidence')
eket, ekpo, ekko, ekbe, ekes = silver('EKET'), silver('EKPO'), silver('EKKO'), silver('EKBE'), silver('EKES')
_nz   = lambda c: F.regexp_replace(F.col(c).cast('string'), '^0+', '')
# the POs of the extract: created on or after CREATED_FROM
pos = ekko.filter(F.col('Created_On') >= F.lit(CREATED_FROM)).select('Purchasing_Document', 'Created_On', 'Purch_Doc_Category')
# goods receipts per PO item (physical GR: VGABE 1 / BEWTP E), as cell 20 of NB_Gold_PROC counts them
gr_item = (ekbe.filter(F.col('Transevent_Type').eqNullSafe('1') | F.col('PO_History_Category').eqNullSafe('E'))
           .groupBy('Purchasing_Document', 'Item')
           .agg(F.sum(F.col('Signed_Quantity').cast('double')).alias('Item_Gr_Qty'), F.max('Posting_Date').alias('Last_GR_Date'),
                F.count('*').alias('GR_Rows')))
for _n, _df in [('fact', fact), ('ev', ev), ('eket', eket), ('ekpo', ekpo), ('ekko', ekko), ('ekbe', ekbe), ('ekes', ekes),
                ('pos', pos), ('gr_item', gr_item)]:
    _df.createOrReplaceTempView(_n)
print(f"Fact_PO_LineItem_Schedule: {fact.count():,} rows | POs created since {CREATED_FROM}: {pos.count():,}")
_ex = ', '.join(f"'{p}'" for p in EXAMPLE_POS)

StatementMeta(, d128d76f-41af-48d1-bc01-8de74e6840cc, 18, Finished, Available, Finished, False)

Pipeline : PROC_Check_Feedback_V4
Run ID   : d4e15b9f-b9f0-4214-a9c5-4e4ef06b7bc5
Started  : 2026-11-19 20:51:33 UTC
Locations:
  bronze     dbo              abfss://3532debe-0172-4493-815c-dc7aed901ac7@onelake.dfs.fabric.micros…
  silver     dbo              abfss://3532debe-0172-4493-815c-dc7aed901ac7@onelake.dfs.fabric.micros…
  gold       dbo              abfss://3532debe-0172-4493-815c-dc7aed901ac7@onelake.dfs.fabric.micros…
  md_silver  dbo              abfss://9cb5a2e3-2121-4f0d-8289-c1f350ac646f@onelake.dfs.fabric.micros…
  md_gold    dbo              abfss://9cb5a2e3-2121-4f0d-8289-c1f350ac646f@onelake.dfs.fabric.micros…
  ops        dbo              abfss://548c95f1-2c7c-44e4-a26f-e6971f80339f@onelake.dfs.fabric.micros…
  sd_silver  dbo              abfss://303715d4-7108-4456-a1f2-bbd04c28cc0c@onelake.dfs.fabric.micros…
  xrate      Exchange_Rates   abfss://9cb5a2e3-2121-4f0d-8289-c1f350ac646f@onelake.dfs.fabric.micros…
Fact_PO_LineItem_Schedule: 6,914,717 rows | POs created 

### Q1 · D1 — lines with no goods receipt in Fabric
**Issue:** 85,777 lines are Open in Fabric and Closed by GR in Every Angle; all have EKET WEMNG 0 and no EKBE row. **Expected:** about the same number here if Fabric lacks the receipts. **Feeds:** whether Amarildo and team must fix the EKET / EKBE extract.

In [6]:
# ── Q1 · D1: schedule lines with no goods receipt in Fabric, by month and plant ────────────────────────────────
# Every Angle closes them by GR. Expected: about 86,000 lines (the extract) when Fabric lacks the receipts; far fewer when the
# extract was taken from an old run and Fabric has caught up since.
q1 = spark.sql(f'''
SELECT date_format(p.Created_On, 'yyyy-MM') AS created_month, i.Plant,
       SUM(CASE WHEN COALESCE(l.Quantity_Delivered, 0) = 0 AND g.Item_Gr_Qty IS NULL AND COALESCE(l.Scheduled_Quantity, 0) > 0 THEN 1 ELSE 0 END) AS lines_no_receipt,
       SUM(CASE WHEN COALESCE(l.Quantity_Delivered, 0) > 0 OR g.Item_Gr_Qty IS NOT NULL THEN 1 ELSE 0 END)   AS lines_with_receipt,
       COUNT(*) AS lines
FROM eket l JOIN pos p ON p.Purchasing_Document = l.Purchasing_Document
LEFT JOIN ekpo i ON i.Purchasing_Document = l.Purchasing_Document AND i.Item = l.Item
LEFT JOIN gr_item g ON g.Purchasing_Document = l.Purchasing_Document AND g.Item = l.Item
GROUP BY 1, 2 ORDER BY 1, 2''')
display(q1)
_tot = q1.agg(F.sum('lines_no_receipt'), F.sum('lines_with_receipt')).first()
print(f'Lines without any receipt in Fabric: {_tot[0] or 0:,} | with a receipt: {_tot[1] or 0:,}')

StatementMeta(, d128d76f-41af-48d1-bc01-8de74e6840cc, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8027965d-7792-4a1f-a2ab-c0667aa27093)

Lines without any receipt in Fabric: 65,502 | with a receipt: 236,991


### Q2 · D1 / D10 — Bronze freshness
**Issue:** goods receipts update existing EKET rows and add EKBE rows. **Expected:** recent `_UPDATED_AT`, `rows_changed > 0` for EKET. **Feeds:** extract diagnosis.

In [7]:
# ── Q2 · D1 / D10: are Bronze EKKO, EKET and EKBE still being refreshed? ──────────────────────────────────────
# Expected: _UPDATED_AT close to the last run for all three; rows_changed > 0 for EKET on recent runs (goods receipts update
# WEMNG on existing EKET rows). A table whose rows_changed is always 0 only receives inserts.
for _t in ['EKKO', 'EKET', 'EKBE']:
    _b = read_table('bronze', _t)
    _r = _b.agg(F.count('*'), F.max('_UPDATED_AT'), F.sum(F.col('_IS_DELETED').cast('int'))).first()
    print(f'Bronze {_t}: {_r[0]:,} rows, last _UPDATED_AT {_r[1]}, deleted {_r[2] or 0:,}')
    try:
        display(spark.sql(f"DESCRIBE HISTORY delta.`{tpath('bronze', _t)}`").select('version', 'timestamp', 'operation', 'operationMetrics').limit(5))
    except Exception as e:
        print(f'  history not readable ({str(e)[:100]})')
try:
    _log = read_table('ops', 'table_log')
    display(_log.filter(F.col('table_name').isin('EKKO', 'EKET', 'EKBE') & (F.col('logged_at') >= F.lit('2026-11-01')))
                .select('logged_at', 'layer', 'table_name', 'status', 'load_mode', 'rows_source', 'rows_in', 'rows_new', 'rows_changed',
                        'rows_deleted', 'rows_out', 'duration_s')
                .orderBy(F.desc('logged_at')))
except Exception as e:
    print(f'OPS table_log not readable ({str(e)[:120]})')

StatementMeta(, d128d76f-41af-48d1-bc01-8de74e6840cc, 20, Finished, Available, Finished, False)

Bronze EKKO: 3,401,432 rows, last _UPDATED_AT 2026-11-19 08:18:15.750734, deleted 0


SynapseWidget(Synapse.DataFrame, f595643c-c74f-424b-991e-9677b88fd32a)

Bronze EKET: 6,914,732 rows, last _UPDATED_AT 2026-11-19 08:15:36.154512, deleted 0


SynapseWidget(Synapse.DataFrame, 68d291f6-f6c5-4a85-9393-db3bf3fe88de)

Bronze EKBE: 17,889,092 rows, last _UPDATED_AT 2026-11-19 08:18:29.050113, deleted 0


SynapseWidget(Synapse.DataFrame, 596df164-9e9b-46c9-85ec-a3deda6ee5a9)

SynapseWidget(Synapse.DataFrame, 1797cd7d-438d-404b-b162-c135c66d27e3)

### Q3 · D1 — example POs through every layer
**Issue:** where does the receipt disappear. **Expected:** Bronze already lacks it (source gap). **Feeds:** who fixes it.

In [8]:
# ── Q3 · D1: the example POs through every layer ─────────────────────────────────────────────────────────────
# Expected (if D1 is a source gap): EKET WEMNG 0 and no EKBE rows in Bronze already. If Bronze has the receipt and Silver or
# Gold does not, the loss is ours (then see Q4).
_br_eket = read_table('bronze', 'EKET').filter(_nz('EBELN').isin([p.lstrip('0') for p in EXAMPLE_POS]))
display(_br_eket.select('EBELN', 'EBELP', 'ETENR', 'EINDT', 'MENGE', 'WEMNG', 'DABMG', 'MNG02', '_UPDATED_AT', '_IS_DELETED').orderBy('EBELN', 'EBELP', 'ETENR'))
_br_ekbe = read_table('bronze', 'EKBE').filter(_nz('EBELN').isin([p.lstrip('0') for p in EXAMPLE_POS]))
print(f'Bronze EKBE rows for the examples: {_br_ekbe.count():,}')
display(_br_ekbe.select('EBELN', 'EBELP', 'VGABE', 'BEWTP', 'BWART', 'BUDAT', 'MENGE', 'SHKZG', 'ETENS', '_UPDATED_AT').orderBy('EBELN', 'EBELP', 'BUDAT'))
display(spark.sql(f'''
SELECT l.Purchasing_Document, l.Item, l.Schedule_Line, l.Scheduled_Quantity, l.Quantity_Delivered, l.Qty_Reduced_MRP, l.Committed_Quantity,
       i.Delivery_Completed, g.Item_Gr_Qty, g.GR_Rows, g.Last_GR_Date
FROM eket l LEFT JOIN ekpo i ON i.Purchasing_Document = l.Purchasing_Document AND i.Item = l.Item
LEFT JOIN gr_item g ON g.Purchasing_Document = l.Purchasing_Document AND g.Item = l.Item
WHERE l.Purchasing_Document IN ({_ex}) ORDER BY 1, 2, 3'''))
display(fact.filter(F.col('Purchasing_Document').isin(EXAMPLE_POS))
            .select('Purchasing_Document', 'Item', 'Schedule_Line', 'Order_Status_Purchase', 'Execution_Status', 'Scheduled_Quantity',
                    'Quantity_Delivered', 'Realized_Finish_Date', 'Piece_Count', 'Confirmation_Categories')
            .orderBy('Purchasing_Document', 'Item', 'Schedule_Line'))
if SOURCE_ABFSS:
    for _t, _cols in [('EKET', ['EBELN', 'EBELP', 'ETENR', 'MENGE', 'WEMNG', 'DABMG']), ('EKBE', ['EBELN', 'EBELP', 'VGABE', 'BWART', 'BUDAT', 'MENGE'])]:
        try:
            _s = spark.read.format('delta').load(f'{SOURCE_ABFSS}/Tables/{SOURCE_SCHEMA}/{_t}')
            display(_s.filter(_nz('EBELN').isin([p.lstrip('0') for p in EXAMPLE_POS])).select(*_cols))
        except Exception as e:
            print(f'Source {_t} not readable ({str(e)[:120]})')
else:
    print('SOURCE_ABFSS not set: the DAB source is not compared (Amarildo and team can run the same selection there)')

StatementMeta(, d128d76f-41af-48d1-bc01-8de74e6840cc, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 84680e74-64ff-419c-965c-8af02890f176)

Bronze EKBE rows for the examples: 29


SynapseWidget(Synapse.DataFrame, e7f5f54b-877f-4356-85ad-b3e860eef635)

SynapseWidget(Synapse.DataFrame, 82530fb8-35bd-4829-bd55-74f1e0dfff6a)

SynapseWidget(Synapse.DataFrame, ce56989c-a151-4d5c-b9f9-50b320f60a08)

SOURCE_ABFSS not set: the DAB source is not compared (Amarildo and team can run the same selection there)


### Q4 · D1 — EKBE keys
**Issue:** a leading-zero or type mismatch would drop receipts in Silver / Gold. **Expected:** 0 unmatched, one key length. **Feeds:** a config fix only if this fails.

In [9]:
# ── Q4 · D1: do the EKBE keys match EKPO? ────────────────────────────────────────────────────────────────────
# Expected: 0 unmatched rows and one key length per table. Unmatched rows or two lengths mean a leading-zero / type problem.
display(spark.sql('''
SELECT COUNT(*) AS ekbe_rows,
       SUM(CASE WHEN i.Purchasing_Document IS NULL THEN 1 ELSE 0 END) AS no_ekpo_match
FROM ekbe b LEFT JOIN ekpo i ON i.Purchasing_Document = b.Purchasing_Document AND i.Item = b.Item'''))
for _n in ['ekbe', 'eket', 'ekpo']:
    display(spark.sql(f'''SELECT '{_n}' AS tbl, LENGTH(Purchasing_Document) AS po_len, LENGTH(Item) AS item_len, COUNT(*) AS rows
                          FROM {_n} GROUP BY 1, 2, 3 ORDER BY 4 DESC'''))

StatementMeta(, d128d76f-41af-48d1-bc01-8de74e6840cc, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f551ac0a-0169-4f33-93a5-d9361c8c1ba6)

SynapseWidget(Synapse.DataFrame, b0552145-1526-4fad-b09c-485f09f4d7da)

SynapseWidget(Synapse.DataFrame, 249f0674-2c11-4a2b-8d2b-d06be0d81fdf)

SynapseWidget(Synapse.DataFrame, 62d08eff-e66c-4f12-9106-22ea0d8683e2)

### Q5 · D3 — zero-quantity schedule lines
**Issue:** 27,392 items end with a line of quantity 0 that Every Angle hides; 7,940 of those lines show a receipt. **Expected:** the committed quantity equals the item's scheduled quantity; receipts on zero lines explained. **Feeds:** business decision D3.

In [10]:
# ── Q5 · D3: zero-quantity schedule lines ────────────────────────────────────────────────────────────────────
# Expected: on most items the zero line is the last one and its committed quantity equals the other lines' scheduled quantity
# (an EKET row carrying the ATP commitment; Every Angle hides it). A zero line with WEMNG > 0 should not exist: list them.
_zl = spark.sql('''
WITH l AS (SELECT Purchasing_Document, Item, Schedule_Line, COALESCE(Scheduled_Quantity, 0) AS sched, COALESCE(Quantity_Delivered, 0) AS recv,
                  COALESCE(Committed_Quantity, 0) AS com, MAX(Schedule_Line) OVER (PARTITION BY Purchasing_Document, Item) AS last_line,
                  SUM(COALESCE(Scheduled_Quantity, 0)) OVER (PARTITION BY Purchasing_Document, Item) AS item_sched
           FROM eket)
SELECT COUNT(*) AS zero_lines,
       SUM(CASE WHEN Schedule_Line = last_line THEN 1 ELSE 0 END)            AS zero_is_last_line,
       SUM(CASE WHEN ABS(com - item_sched) < 0.001 THEN 1 ELSE 0 END)        AS committed_equals_item_scheduled,
       SUM(CASE WHEN recv > 0 THEN 1 ELSE 0 END)                             AS zero_lines_with_receipt
FROM l WHERE sched = 0 AND com > 0''')
display(_zl)
display(spark.sql(f'''
SELECT l.Purchasing_Document, l.Item, l.Schedule_Line, l.Scheduled_Quantity, l.Quantity_Delivered, l.Committed_Quantity, l.Committed_Date,
       l.Delivery_Date, g.Item_Gr_Qty, g.GR_Rows
FROM eket l LEFT JOIN gr_item g ON g.Purchasing_Document = l.Purchasing_Document AND g.Item = l.Item
WHERE COALESCE(l.Scheduled_Quantity, 0) = 0 AND COALESCE(l.Quantity_Delivered, 0) > 0
ORDER BY l.Purchasing_Document, l.Item LIMIT {N_EXAMPLES}'''))

# ▶ 18 Nov 2026 (first run): 720,937 zero lines show a receipt. In 13219607 item 2 both lines carry WEMNG 1,200 and EKBE has 1,200,
#   so adding Quantity_Delivered over an item's lines counts the receipt twice. How many items does that affect?
display(spark.sql('''
WITH s AS (SELECT Purchasing_Document, Item, SUM(COALESCE(Quantity_Delivered, 0)) AS line_recv,
                  MAX(CASE WHEN COALESCE(Scheduled_Quantity, 0) = 0 AND COALESCE(Quantity_Delivered, 0) > 0 THEN 1 ELSE 0 END) AS zero_line_recv
           FROM eket GROUP BY Purchasing_Document, Item)
SELECT s.zero_line_recv = 1 AS has_zero_line_with_receipt, COUNT(*) AS items,
       SUM(CASE WHEN g.Item_Gr_Qty IS NOT NULL AND s.line_recv > g.Item_Gr_Qty + 0.001 THEN 1 ELSE 0 END) AS items_lines_above_ekbe_gr,
       SUM(CASE WHEN g.Item_Gr_Qty IS NOT NULL AND ABS(s.line_recv - 2 * g.Item_Gr_Qty) <= 0.001 AND g.Item_Gr_Qty > 0 THEN 1 ELSE 0 END) AS items_lines_twice_ekbe_gr
FROM s LEFT JOIN gr_item g ON g.Purchasing_Document = s.Purchasing_Document AND g.Item = s.Item
GROUP BY 1 ORDER BY 1'''))


StatementMeta(, d128d76f-41af-48d1-bc01-8de74e6840cc, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 77a7c75d-1291-4f78-8b9b-cb5fd6fb7d40)

SynapseWidget(Synapse.DataFrame, ef66b976-2428-474d-81e9-35f14f0c5ae4)

SynapseWidget(Synapse.DataFrame, cec526ca-ffe9-4d3d-897c-263d28bac78f)

### Q6 · D5 — Closed by GR vs Outbound delivery created
**Issue:** 8,439 fully received stock-transfer lines. **Expected:** PSTYP U with a delivery. **Feeds:** rule order for stock transfers (business).

In [11]:
# ── Q6 · D5: Closed by GR lines on stock transfers (Every Angle: Outbound delivery created) ───────────────────
# Expected: PSTYP U items, fully received, with an outbound delivery (LIPS) or an inbound delivery on the confirmation. The
# question for the business is the rule order for these lines (rule 2 Closed by GR before rule 10 today).
_sto = (fact.filter(F.col('Order_Status_Purchase') == 'Closed by GR')
            .join(ekpo.select('Purchasing_Document', 'Item', 'Item_Category_Ext'), ['Purchasing_Document', 'Item'], 'left')   # Delivery_Completed is on the fact already (18 Nov 2026)
            .filter(F.trim(F.col('Item_Category_Ext')).eqNullSafe('U')))
print(f"Closed by GR on stock-transfer items (PSTYP U): {_sto.count():,} lines")
_mv = ekbe.groupBy('Purchasing_Document', 'Item').agg(F.concat_ws('|', F.sort_array(F.collect_set('Movement_Type'))).alias('Movement_Types'))
_ib = ekes.groupBy('Purchasing_Document', 'Item').agg(F.max(F.when(F.trim(F.coalesce(F.col('Inbound_Delivery'), F.lit(''))) != '', F.col('Inbound_Delivery'))).alias('Inbound_Delivery'))
_q6 = (_sto.join(_mv, ['Purchasing_Document', 'Item'], 'left').join(_ib, ['Purchasing_Document', 'Item'], 'left'))
if 'sd_silver' in LOC and table_exists('sd_silver', 'LIPS'):
    _lips = (read_table('sd_silver', 'LIPS', active=True).select(_nz('Reference_Document').alias('Purchasing_Document'), _nz('Reference_Item').alias('Item'))
             .dropDuplicates().withColumn('Has_LIPS_Delivery', F.lit(True)))
    _q6 = _q6.join(_lips, ['Purchasing_Document', 'Item'], 'left')
else:
    print('SD Silver LIPS not reachable: Has_LIPS_Delivery not shown')
    _q6 = _q6.withColumn('Has_LIPS_Delivery', F.lit(None).cast('boolean'))
display(_q6.join(pos.select('Purchasing_Document'), 'Purchasing_Document')   # ▶ 18 Nov 2026: examples from the extract's POs
           .select('Purchasing_Document', 'Item', 'Schedule_Line', 'Execution_Status', 'Scheduled_Quantity', 'Quantity_Delivered',
                   'Qty_Delivered_STO', 'Committed_Quantity', 'Delivery_Completed', 'Movement_Types', 'Inbound_Delivery', 'Has_LIPS_Delivery')
           .orderBy('Purchasing_Document', 'Item', 'Schedule_Line').limit(N_EXAMPLES))

StatementMeta(, d128d76f-41af-48d1-bc01-8de74e6840cc, 24, Finished, Available, Finished, False)

Closed by GR on stock-transfer items (PSTYP U): 2,174,989 lines


SynapseWidget(Synapse.DataFrame, fc3d494a-f992-46bd-b91c-b95b10bee93b)

### Q7 · D6 — open-line status pairs
**Issue:** four pairs where both sides see an open line but name it differently. **Expected:** the EKES rows and committed quantities show which evidence Every Angle used. **Feeds:** business decisions on rules 10, 11, 14, 15.

In [12]:
# ── Q7 · D6: the four open-line status pairs — what evidence each side sees ─────────────────────────────────
# Fabric → Every Angle: To be confirmed → Stock committed (2,003 in the extract); Stock committed → Outbound delivery created
# (1,081); Shipping notification → Confirmed (945); To be confirmed → Outbound delivery created (1,166). For N_EXAMPLES lines of
# each Fabric status: the item's EKES rows (category, dates, quantities), committed quantity and date, PSTYP, BSTAE / KZABS / LABNR.
_conf = (ekes.filter(F.trim(F.coalesce(F.col('Confirmation_Deletion'), F.lit(''))) == '')
             .groupBy('Purchasing_Document', 'Item')
             .agg(F.count('*').alias('EKES_Rows'),
                  F.concat_ws('|', F.sort_array(F.collect_set('Confirmation_Category'))).alias('EKES_Categories'),
                  F.max('Confirmed_Delivery_Date').alias('Last_Confirmed_Date'), F.max('Created_On').alias('Last_Conf_Created'),
                  F.sum(F.col('Confirmation_Quantity').cast('double')).alias('Conf_Qty'),
                  F.sum(F.col('Confirmation_Qty_Reduced_MRP').cast('double')).alias('Conf_Qty_Reduced'),
                  F.max(F.when(F.trim(F.coalesce(F.col('Inbound_Delivery'), F.lit(''))) != '', F.col('Inbound_Delivery'))).alias('Inbound_Delivery')))
_itm = ekpo.select('Purchasing_Document', 'Item', 'Item_Category_Ext', 'Confirmation_Control', 'Order_Acknowledgment')   # Acknowledgment_Reqd is on the fact already (18 Nov 2026)
for _status in ['To be confirmed', 'Stock committed', 'Shipping notification']:
    print(f'\n{_status}: open lines, {N_EXAMPLES} examples')
    display(fact.filter((F.col('Order_Status_Purchase') == _status) & (F.coalesce(F.col('Quantity_Delivered'), F.lit(0)) == 0))
                .join(pos.select('Purchasing_Document'), 'Purchasing_Document')   # ▶ 18 Nov 2026: the extract's POs (the first run showed 2018-2022 lines)
                .join(_conf, ['Purchasing_Document', 'Item'], 'left').join(_itm, ['Purchasing_Document', 'Item'], 'left')
                .select('Purchasing_Document', 'Item', 'Schedule_Line', 'Delivery_Date', 'Scheduled_Quantity', 'Committed_Quantity', 'Committed_Date',
                        'Item_Category_Ext', 'Confirmation_Control', 'Acknowledgment_Reqd', 'Order_Acknowledgment',
                        'EKES_Rows', 'EKES_Categories', 'Last_Confirmed_Date', 'Last_Conf_Created', 'Conf_Qty', 'Conf_Qty_Reduced', 'Inbound_Delivery',
                        'Piece_Count', 'Confirmation_Categories')
                .orderBy('Purchasing_Document', 'Item', 'Schedule_Line').limit(N_EXAMPLES))

StatementMeta(, d128d76f-41af-48d1-bc01-8de74e6840cc, 25, Finished, Available, Finished, False)


To be confirmed: open lines, 20 examples


SynapseWidget(Synapse.DataFrame, 66a7a36e-7aa4-48d3-b996-7c86bd049551)


Stock committed: open lines, 20 examples


SynapseWidget(Synapse.DataFrame, ab0d5719-3633-417e-87c9-2e6acf30b3b8)


Shipping notification: open lines, 20 examples


SynapseWidget(Synapse.DataFrame, 10ef21aa-7b95-49e6-a2ce-245c2eaf90b5)

### Q8 · D10 — missing PO
**Issue:** PO 13148876 (created 24 Sep) is in Every Angle, not in Fabric. **Expected:** absent from Bronze or flagged deleted. **Feeds:** Amarildo and team.

In [13]:
# ── Q8 · D10: the PO Every Angle has and Fabric lacks ───────────────────────────────────────────────────────
# Expected: no row in Bronze EKKO (extract gap) or a row with _IS_DELETED = true (deleted upstream). The business should send
# the full list of missing POs; EXAMPLE_POS has the one example from the file (13148876).
for _t, _key in [('EKKO', 'EBELN'), ('EKPO', 'EBELN'), ('EKET', 'EBELN')]:
    _b = read_table('bronze', _t).filter(_nz(_key).isin([p.lstrip('0') for p in EXAMPLE_POS]))
    display(_b.groupBy(_key, '_IS_DELETED').agg(F.count('*').alias('rows'), F.min('_CREATED_AT').alias('first_loaded'), F.max('_UPDATED_AT').alias('last_updated'))
              .withColumn('table', F.lit(_t)).orderBy(_key))

StatementMeta(, d128d76f-41af-48d1-bc01-8de74e6840cc, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4cc0999f-8f7c-44ff-863c-30d20e72a891)

SynapseWidget(Synapse.DataFrame, 18006137-384a-44d0-aece-8cd6f13c4914)

SynapseWidget(Synapse.DataFrame, cf77d082-11c4-420e-a3f9-9590f8de8a7d)

### Q9 · D7 / D8 — duplicates and over-confirmed pieces
**Expected after the 17 Nov change:** 0 keys with more than one row, 0 rows without a schedule line; the pieces are in the evidence table.

In [14]:
# ── Q9 · D7 / D8: duplicated keys and over-confirmed pieces ─────────────────────────────────────────────────
# Before 17 Nov 2026 the fact was one row per segment: a partly confirmed line had two rows (259 keys in the extract) and the
# over-confirmed pieces had no schedule line (4,398 rows). Expected now: 0 and 0 in the fact; the pieces are in the evidence table.
_dup = fact.groupBy('Purchasing_Document', 'Item', 'Schedule_Line').count().filter('count > 1')
print(f"Keys with more than one row in the fact: {_dup.count():,} (expected 0) | rows with an empty Schedule_Line: "
      f"{fact.filter(F.col('Schedule_Line').isNull()).count():,} (expected 0)")
print(f"Evidence table: {ev.count():,} segments; over-confirmed pieces (no schedule line): "
      f"{ev.filter(F.coalesce(F.col('Is_Over_Confirmed'), F.lit(False))).count():,}; lines with more than one piece: "
      f"{fact.filter(F.col('Piece_Count') > 1).count():,}")
display(fact.filter(F.col('Piece_Count') > 1)
            .select('Purchasing_Document', 'Item', 'Schedule_Line', 'Order_Status_Purchase', 'Piece_Count', 'Confirmation_Categories', 'Segment_Quantity')
            .orderBy('Purchasing_Document', 'Item', 'Schedule_Line').limit(N_EXAMPLES))

StatementMeta(, d128d76f-41af-48d1-bc01-8de74e6840cc, 27, Finished, Available, Finished, False)

Keys with more than one row in the fact: 0 (expected 0) | rows with an empty Schedule_Line: 0 (expected 0)
Evidence table: 6,934,450 segments; over-confirmed pieces (no schedule line): 18,414; lines with more than one piece: 631


SynapseWidget(Synapse.DataFrame, bdcc351f-1018-482c-a6e6-04e19a7b4572)

### Q10 · D2 — line-level execution status
**Issue:** 9,292 lines fully received show Partially Open because the status is the item's. **Expected:** those lines become Closed. **Feeds:** business decision D2.

In [15]:
# ── Q10 · D2: a line-level execution status against the item-level one ───────────────────────────────────────
# Proposal D2 (NB_Gold_PROC cell 22, commented): Cancelled when the item is; Closed when ELIKZ is set or nothing is left to receive;
# Partially Open when some is received; else Open. Expected: the Partially Open → Closed pairs (9,292 in the extract) move.
_recv, _sched = F.coalesce(F.col('Quantity_Delivered'), F.lit(0)), F.coalesce(F.col('Scheduled_Quantity'), F.lit(0))
_line_status = (F.when(F.col('Execution_Status').isin('Cancelled_Header', 'Cancelled_Line'), F.col('Execution_Status'))
                 .when((F.col('Delivery_Completed') == 'X') | ((_sched - _recv) <= OS_TOLERANCE), 'Closed')
                 .when(_recv > 0, 'Partially Open').otherwise('Open'))
_q10 = fact.withColumn('Line_Execution_Status', _line_status)
display(_q10.groupBy('Execution_Status', 'Line_Execution_Status').count().orderBy(F.desc('count')))
print(f"Lines whose status would change: {_q10.filter(F.col('Execution_Status') != F.col('Line_Execution_Status')).count():,} of {fact.count():,}")

StatementMeta(, d128d76f-41af-48d1-bc01-8de74e6840cc, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 202c534e-cfd1-4b9a-91b8-48febf5d303a)

Lines whose status would change: 8,388 of 6,914,717


### Q11 · D1 — time pattern of the missing receipts
**Issue:** are receipts missing since one date (a cut-off) or here and there (sporadic)? **Expected:** a steady weekly count of goods receipts and of changed EKET rows means sporadic; a sudden drop means a cut-off from that week. Compare the weekly counts with the source / Every Angle. **Feeds:** what Amarildo and team look for in the DAB extract.

In [16]:
# ── Q11 · D1: goods receipts per posting week, and Bronze rows changed per day (18 Nov 2026) ──────────────────────
# Physical GR (VGABE 1 / BEWTP E) in Silver EKBE by posting week since 1 Aug, and the share of items of the extract's POs that have one.
_gr = ekbe.filter(F.col('Transevent_Type').eqNullSafe('1') | F.col('PO_History_Category').eqNullSafe('E'))
display(_gr.filter(F.col('Posting_Date') >= F.lit('2026-08-01'))
           .groupBy(F.date_trunc('week', F.col('Posting_Date')).cast('date').alias('posting_week'))
           .agg(F.count('*').alias('gr_rows'), F.countDistinct('Purchasing_Document', 'Item').alias('items_with_gr'),
                F.max('Posting_Date').alias('last_posting_date'))
           .orderBy('posting_week'))
# When Bronze last received each row: new or changed EKET / EKBE rows per load day (a gap of days = no load; a low count = little change)
for _t in ('EKET', 'EKBE'):
    print(f'Bronze {_t}: rows by the day they were last loaded or changed (since 1 Sep)')
    display(read_table('bronze', _t).filter(F.col('_UPDATED_AT') >= F.lit('2026-09-01'))
                .groupBy(F.to_date('_UPDATED_AT').alias('load_day')).count().orderBy('load_day'))


StatementMeta(, d128d76f-41af-48d1-bc01-8de74e6840cc, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e4721b10-605f-4a75-96bb-44e7d055217d)

Bronze EKET: rows by the day they were last loaded or changed (since 1 Sep)


SynapseWidget(Synapse.DataFrame, 2111282d-c105-498b-8083-4207f9e48732)

Bronze EKBE: rows by the day they were last loaded or changed (since 1 Sep)


SynapseWidget(Synapse.DataFrame, 8365d965-2ed9-4e33-84db-f048d0e9b494)

### Q12 · line-level comparison with the 16 Nov extract
`Procurment/PROC_EA_16Nov_upload.csv` holds one row per line of the 16 Nov extract (289,398 lines): Every Angle's order and execution
status, Fabric's 16 Nov values and the 16 Nov cause (1–13, as in `PROC_Order_Status_Differences.xlsx`). Upload it to the PROC Gold
lakehouse under `Files/checks` (Lakehouse explorer → Files → Upload). Q12 joins it to today's `Fact_PO_LineItem_Schedule`, re-derives the
cause, adds the execution status with Option A (`NB_Gold_PROC` cell 13, 19 Nov 2026) and prints everything as text, so the exported
notebook carries the results. With `SAVE_COMPARE_TABLE = True` the full line list is also written to Gold `Chk_PROC_Order_Status_Compare`
for SQL queries.

In [17]:
# ── Q12 · line-level comparison: 16 Nov extract (Every Angle + Fabric then) against Fabric now, with Option A (19 Nov 2026) ──────────
# Option A is re-derived from Silver EKET so the result shows its effect before and after NB_Gold_PROC cell 13 runs with it: Delivered
# adds WEMNG only over lines with a quantity (a zero line keeps its WEMNG only when it holds more than that sum); an item whose lines are
# all zero keeps the sum over all its lines. Last_GR_Date after 16 Nov = SAP changed after the extract, so EA may have moved too.
_root = LOC['gold'][0]
_root = _root[:_root.rfind('/Tables')] if '/Tables' in _root else _root
_ea = spark.read.option('header', True).csv(f'{_root}/{EA_FILE}')
_k  = lambda df, po, it, sl: df.withColumn('_po', F.regexp_replace(F.trim(F.col(po)), '^0+', '')) \
                               .withColumn('_it', F.regexp_replace(F.trim(F.col(it)), '^0+', '')) \
                               .withColumn('_sl', F.regexp_replace(F.trim(F.col(sl)), '^0+', ''))
_ea = _k(_ea, 'PO', 'Item', 'Schedule_Line')
_has = F.coalesce(F.col('Scheduled_Quantity'), F.lit(0)) != 0
_item = (eket.groupBy('Purchasing_Document', 'Item')
             .agg(F.sum('Quantity_Delivered').alias('Item_Delivered_All_Lines'),
                  F.sum(F.when(_has, F.col('Quantity_Delivered'))).alias('_q'),
                  F.max(F.when(~_has, F.col('Quantity_Delivered'))).alias('_z'),
                  F.max(_has.cast('int')).alias('_any'),
                  F.sum('Scheduled_Quantity').alias('Item_Scheduled_Qty'))
             .withColumn('Item_Delivered_A', F.when(F.col('_any') == 1, F.greatest('_q', '_z')).otherwise(F.col('Item_Delivered_All_Lines')))
             .drop('_q', '_z', '_any'))
_now = (fact.select('Purchasing_Document', 'Item', 'Schedule_Line', F.col('Order_Status_Purchase').alias('Fabric_Order_Status_Now'),
                    F.col('Execution_Status').alias('_ex_now'))
            .join(eket.select('Purchasing_Document', 'Item', 'Schedule_Line', 'Scheduled_Quantity', 'Quantity_Delivered'),
                  ['Purchasing_Document', 'Item', 'Schedule_Line'], 'left')
            .join(_item, ['Purchasing_Document', 'Item'], 'left')
            .join(ekpo.select('Purchasing_Document', 'Item', 'Delivery_Completed'), ['Purchasing_Document', 'Item'], 'left')
            .join(gr_item.select('Purchasing_Document', 'Item', 'Item_Gr_Qty', 'Last_GR_Date'), ['Purchasing_Document', 'Item'], 'left')
            .join(ekko.select('Purchasing_Document', F.col('Purch_Doc_Category').alias('Header_Category')), 'Purchasing_Document', 'left'))
_dA, _sq = F.col('Item_Delivered_A'), F.col('Item_Scheduled_Qty')
_exn = lambda c: (F.when(c.startswith('Cancelled'), 'Cancelled').when(F.lower(c) == 'partially open', 'Partially open').otherwise(c))
_now = (_now.withColumn('_ex_A', F.when(F.col('_ex_now').startswith('Cancelled'), F.col('_ex_now'))
                                  .when(F.col('Delivery_Completed') == 'X', 'Closed')
                                  .when((_dA > 0) & (_dA < _sq), 'Partially Open').when(_dA >= _sq, 'Closed').otherwise('Open'))
            .withColumn('Fabric_Execution_Now', _exn(F.col('_ex_now'))).withColumn('Fabric_Execution_A', _exn(F.col('_ex_A')))
            .drop('_ex_now', '_ex_A'))
_now = _k(_now, 'Purchasing_Document', 'Item', 'Schedule_Line')
cmp_ = _ea.join(_now.drop('Purchasing_Document', 'Item', 'Schedule_Line'), ['_po', '_it', '_sl'], 'left')
_f, _e, _o = F.col('Fabric_Order_Status_Now'), F.col('EA_Order_Status'), F.col('Fabric_Order_Status_16Nov')
_OPEN = ['To be confirmed', 'Confirmed', 'Ordered', 'Shipping notification', 'Goods issued', 'Outbound delivery created', 'Stock committed',
         'GR blocked stock (103)', 'GR blocked stock', 'Blocked', 'Service entry sheet created']
_CANC, _OC = ['Cancelled completely', 'Cancelled partially'], 'Closed by order creation or IR'
_sc, _rc, _gr = (F.coalesce(F.col(c).cast('double'), F.lit(0.0)) for c in ('Scheduled_Quantity', 'Quantity_Delivered', 'Item_Gr_Qty'))
_cause_now = (F.when(_f.isNull() | (_f == _e), F.lit(None))          # the same 13 causes as PROC_Order_Status_Differences.xlsx
    .when((F.col('Header_Category') == 'A') | (_f == 'To be ordered'), 10)
    .when((_sc == 0) & ((_e == '#N/A') | ((_f == _OC) & _e.isin(_OPEN))), 2)
    .when(_e == '#N/A', 13).when(_f == 'Return item', 11)
    .when((_e == 'Closed by GR') & (_f != 'Closed by GR') & (_rc == 0) & (_gr == 0), 1)
    .when((_e == 'Closed by GR') & _f.isin(_OPEN), 8)
    .when((_f == 'Closed by GR') & (_e == 'Outbound delivery created'), 4)
    .when(_f.isin(_CANC) | _e.isin(_CANC), 7)
    .when(((_f == 'Closed by GR') & (_e == _OC)) | ((_f == _OC) & (_e == 'Closed by GR')), 9)
    .when(_f.isin(_OPEN) & (_e == _OC), 6).when(_e == 'To be ordered', 12)
    .when(_f.isin(_OPEN) & _e.isin(_OPEN), 3).otherwise(13))
cmp_ = (cmp_.withColumn('Cause_Now', _cause_now)
    .withColumn('Result', F.when(_f.isNull(), 'No longer in Fabric').when((_o == _e) & (_f == _e), 'Same, unchanged')
                           .when(_f == _e, 'Now matches EA').when(_o == _e, 'Newly different').otherwise('Still different'))
    .withColumn('Receipt_After_16Nov', F.col('Last_GR_Date') > F.lit('2026-11-16'))
    .withColumn('Cause_16Nov', F.col('Cause_16Nov').cast('int'))).cache()
_n_new = _now.join(_ea.select('_po', '_it', '_sl'), ['_po', '_it', '_sl'], 'left_anti') \
             .join(ekko.filter((F.col('Created_On') >= F.lit(CREATED_FROM)) & (F.col('Created_On') <= F.lit(EXTRACT_TO)))
                       .select(F.regexp_replace('Purchasing_Document', '^0+', '').alias('_po')), '_po').count()
_tot = cmp_.agg(F.count('*').alias('n'), F.sum((_o == _e).cast('int')).alias('same_16'),
                F.sum(_f.isNotNull().cast('int')).alias('in_fabric'), F.sum((_f == _e).cast('int')).alias('same_now')).first()
print(f"Q12 · lines in the extract: {_tot['n']:,} | still in Fabric: {_tot['in_fabric']:,} | new lines on the same POs: {_n_new:,}")
print(f"Order status = EA: 16 Nov {_tot['same_16']:,} ({_tot['same_16'] / _tot['n']:.1%}) | now {_tot['same_now']:,} "
      f"({_tot['same_now'] / max(_tot['in_fabric'], 1):.1%} of the lines still in Fabric)")
_ok = F.col('EA_Execution_Line') != '#N/A'
_x = cmp_.filter(_ok & _f.isNotNull()).agg(F.count('*').alias('n'),
        F.sum((F.col('Fabric_Execution_16Nov') == F.col('EA_Execution_Line')).cast('int')).alias('m16'),
        F.sum((F.col('Fabric_Execution_Now') == F.col('EA_Execution_Line')).cast('int')).alias('mnow'),
        F.sum((F.col('Fabric_Execution_A') == F.col('EA_Execution_Line')).cast('int')).alias('mA')).first()
print(f"Execution status = EA (line), {_x['n']:,} lines: 16 Nov {_x['m16']:,} ({_x['m16'] / max(_x['n'], 1):.1%}) | now {_x['mnow']:,} "
      f"({_x['mnow'] / max(_x['n'], 1):.1%}) | with Option A {_x['mA']:,} ({_x['mA'] / max(_x['n'], 1):.1%})")
print('\n[Q12-1] Result per line'); cmp_.groupBy('Result').count().orderBy(F.desc('count')).show(truncate=False)
print('[Q12-2] Per 16 Nov cause')
(cmp_.filter(F.col('Cause_16Nov').isNotNull()).groupBy('Cause_16Nov', 'Cause_16Nov_Name')
     .agg(F.count('*').alias('lines_16Nov'), F.sum((F.col('Result') == 'Now matches EA').cast('int')).alias('now_match'),
          F.sum((F.col('Result') == 'Still different').cast('int')).alias('still_diff'),
          F.sum(((F.col('Result') == 'Still different') & (F.col('Cause_Now') != F.col('Cause_16Nov'))).cast('int')).alias('moved_cause'),
          F.sum((F.col('Result') == 'No longer in Fabric').cast('int')).alias('gone'),
          F.sum(F.coalesce(F.col('Receipt_After_16Nov'), F.lit(False)).cast('int')).alias('receipt_after_16Nov'))
     .orderBy(F.desc('lines_16Nov')).show(20, truncate=False))
print('[Q12-3] Still different and newly different, by cause now')
(cmp_.filter(F.col('Result').isin('Still different', 'Newly different')).groupBy('Cause_Now', 'Result').count()
     .orderBy('Cause_Now', 'Result').show(40, truncate=False))
print('[Q12-4] Fabric order status moves (16 Nov -> now)')
(cmp_.filter(_f.isNotNull() & (_o != _f)).groupBy('Fabric_Order_Status_16Nov', 'Fabric_Order_Status_Now', 'EA_Order_Status').count()
     .orderBy(F.desc('count')).show(40, truncate=False))
print('[Q12-5] Execution status: now -> with Option A, and EA (line)')
(cmp_.filter(_f.isNotNull() & (F.col('Fabric_Execution_Now') != F.col('Fabric_Execution_A')))
     .groupBy('Fabric_Execution_Now', 'Fabric_Execution_A', 'EA_Execution_Line').count().orderBy(F.desc('count')).show(30, truncate=False))
print(f'[Q12-6] Example lines: up to {N_LINE_EXAMPLES} per result and cause')
_w = Window.partitionBy('Result', F.coalesce('Cause_Now', 'Cause_16Nov')).orderBy('_po', '_it', '_sl')
(cmp_.filter(F.col('Result').isin('Now matches EA', 'Still different', 'Newly different', 'No longer in Fabric'))
     .withColumn('_rn', F.row_number().over(_w)).filter(F.col('_rn') <= N_LINE_EXAMPLES)
     .select('Result', 'Cause_16Nov', 'Cause_Now', 'PO', 'Item', 'Schedule_Line', 'EA_Order_Status', 'Fabric_Order_Status_16Nov',
             'Fabric_Order_Status_Now', F.col('Scheduled_Quantity').cast('decimal(18,3)').alias('Sched'),
             F.col('Quantity_Delivered').cast('decimal(18,3)').alias('Recv'), F.col('Item_Gr_Qty').cast('decimal(18,3)').alias('Item_GR'),
             'Last_GR_Date', 'EA_Execution_Line', 'Fabric_Execution_16Nov', 'Fabric_Execution_Now', 'Fabric_Execution_A')
     .orderBy('Result', F.coalesce('Cause_Now', 'Cause_16Nov'), 'PO', 'Item', 'Schedule_Line').show(2000, truncate=False))
print(f'[Q12-7] Example lines whose execution status changes with Option A (up to {N_LINE_EXAMPLES * 3})')
(cmp_.filter(_f.isNotNull() & (F.col('Fabric_Execution_Now') != F.col('Fabric_Execution_A')))
     .select('PO', 'Item', 'Schedule_Line', 'EA_Execution_Line', 'Fabric_Execution_Now', 'Fabric_Execution_A',
             F.col('Item_Delivered_All_Lines').cast('decimal(18,3)').alias('Item_Recv_All'), F.col('Item_Delivered_A').cast('decimal(18,3)').alias('Item_Recv_A'),
             F.col('Item_Scheduled_Qty').cast('decimal(18,3)').alias('Item_Sched'), F.col('Item_Gr_Qty').cast('decimal(18,3)').alias('Item_GR'))
     .orderBy('PO', 'Item', 'Schedule_Line').show(N_LINE_EXAMPLES * 3, truncate=False))
if SAVE_COMPARE_TABLE:
    (cmp_.drop('_po', '_it', '_sl').write.format('delta').mode('overwrite').option('overwriteSchema', 'true')
         .save(tpath('gold', 'Chk_PROC_Order_Status_Compare')))
    register_table('gold', 'Chk_PROC_Order_Status_Compare')
    print('Line list written to Gold Chk_PROC_Order_Status_Compare (drop it when the review is done)')
cmp_.unpersist()

StatementMeta(, d128d76f-41af-48d1-bc01-8de74e6840cc, 30, Finished, Available, Finished, False)

Q12 · lines in the extract: 289,398 | still in Fabric: 289,398 | new lines on the same POs: 271
Order status = EA: 16 Nov 152,092 (52.6%) | now 229,172 (79.2% of the lines still in Fabric)
Execution status = EA (line), 264,434 lines: 16 Nov 156,551 (59.2%) | now 252,692 (95.6%) | with Option A 252,692 (95.6%)

[Q12-1] Result per line
+---------------+------+
|Result         |count |
+---------------+------+
|Same, unchanged|143050|
|Now matches EA |86122 |
|Still different|51184 |
|Newly different|9042  |
+---------------+------+

[Q12-2] Per 16 Nov cause
+-----------+-------------------------------+-----------+---------+----------+-----------+----+-------------------+
|Cause_16Nov|Cause_16Nov_Name               |lines_16Nov|now_match|still_diff|moved_cause|gone|receipt_after_16Nov|
+-----------+-------------------------------+-----------+---------+----------+-----------+----+-------------------+
|1          |Missing receipts               |85663      |84801    |862       |675        |

DataFrame[_po: string, _it: string, _sl: string, PO: string, Item: string, Schedule_Line: string, EA_Order_Status: string, EA_Execution_Line: string, EA_Execution_Header: string, Fabric_Order_Status_16Nov: string, Fabric_Execution_16Nov: string, Cause_16Nov: int, Cause_16Nov_Name: string, Rows_In_Extract: string, Fabric_Order_Status_Now: string, Scheduled_Quantity: decimal(15,3), Quantity_Delivered: decimal(15,3), Item_Delivered_All_Lines: decimal(25,3), Item_Scheduled_Qty: decimal(25,3), Item_Delivered_A: decimal(25,3), Delivery_Completed: string, Item_Gr_Qty: double, Last_GR_Date: date, Header_Category: string, Fabric_Execution_Now: string, Fabric_Execution_A: string, Cause_Now: int, Result: string, Receipt_After_16Nov: boolean]

In [18]:
# ── End-of-run report: result, issues in plain language, tables, checks ────────
end_of_run_report()

StatementMeta(, d128d76f-41af-48d1-bc01-8de74e6840cc, 31, Finished, Available, Finished, True)

════════════════════════════════════════════════════════════════════════════════════════════════════
 PROC_Check_Feedback_V4  |  run d4e15b9f-b9f0-4214-a9c5-4e4ef06b7bc5
 19 Nov 2026 20:51–20:58 UTC (6m 32s)  |  0 of 0 tables OK  |  0 checks
 RESULT: ✓ All 0 tables loaded, all checks passed
════════════════════════════════════════════════════════════════════════════════════════════════════
 Details: SELECT * FROM dq_results WHERE pipeline_run_id = 'd4e15b9f-b9f0-4214-a9c5-4e4ef06b7bc5' ORDER BY status
════════════════════════════════════════════════════════════════════════════════════════════════════
Session stopped to release capacity (END_SESSION = False keeps it)
  Session not stopped (__$$_STOP_SESSION_REQUEST_$$__)


'SUCCEEDED'